# EXP-003+005: BSD35k-CS 데이터 통합 + 계층적 손실함수

## 가설
데이터를 줄이면 안 된다 (EXP-001 교훈: -1%). 반대로 **데이터를 3배 늘리고** **H-Acc를 직접 최적화하는 손실함수**를 쓰면 성능이 뛴다.

## 변경 사항
| 항목 | Baseline (EXP-000) | 이번 실험 |
|------|-------------------|----------|
| 훈련 데이터 | BSD10k ~8,800/fold | BSD10k + BSD35k-CS 필터 ~24,000/fold |
| 손실함수 | CrossEntropy | CE + λ_top × L_Top + λ_contr × L_Contr |
| Seed | 42 | **42** (baseline과 동일) |

## 기댓값
- EXP-003 (데이터): +3~5% H-Acc
- EXP-005 (계층적 손실): +2~3% H-Acc
- 합산: **+5~8% H-Acc → 84~87%** 목표

## 비교 기준
- **Baseline H-Acc: 79.45%** (EXP-000)

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42                        # baseline과 동일
MODE = 'both'
K_FOLDS = 5
NUM_EPOCHS = 100
BATCH_SIZE = 64
LR = 0.001
HIDDEN_SIZE = 128
DROPOUT = 0.1
SCHEDULER_TYPE = 'step'
PATIENCE = 5
EARLY_STOPPING_FACTOR = 3

# BSD35k-CS 필터링
BSD35K_FILTER_THRESHOLD = 0.5    # v4 rank score >= 0.5 → ~15,566 samples (precision ~0.80)
                                  # 0.18 → 26,150개 (더 많지만 노이즈 더 있음)
                                  # 0.7  →  9,197개 (적지만 precision 0.88)

# 계층적 손실 가중치 (논문 B 권장값)
LAMBDA_TOP   = 0.3               # L_Top 가중치
LAMBDA_CONTR = 0.1               # L_Contr 가중치
TEMPERATURE  = 0.07              # contrastive loss temperature

EXP_ID   = 'exp_003'
EXP_DESC = f'BSD35k-CS(thr={BSD35K_FILTER_THRESHOLD}) + HierLoss(λ_top={LAMBDA_TOP}, λ_contr={LAMBDA_CONTR})'
MODEL_OUTPUT_DIR = 'model_output_exp003'

In [ ]:
import sys, os

NOTEBOOK_DIR = os.path.abspath('')
CODE_DIR     = os.path.join(NOTEBOOK_DIR, 'dcase2026_task1_baseline')
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import json, random, subprocess
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from torch.utils.data import DataLoader, ConcatDataset

from models import BaseClassifier
from dataset_utils import HATRDataset
from evaluate import evaluate_model
from losses import CrossEntropyLoss
from utils import set_seed, build_class_to_topclass_mapping

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# 경로
# ============================================================

PROJECT_ROOT = NOTEBOOK_DIR

PROCESSED_CSV      = os.path.join(PROJECT_ROOT, 'dcase2026_task1_baseline', 'data', 'processed_dataset.csv')
CLASS_DICT_JSON    = os.path.join(PROJECT_ROOT, 'dcase2026_task1_baseline', 'data', 'class_dict.json')
TOP_DICT_JSON      = os.path.join(PROJECT_ROOT, 'dcase2026_task1_baseline', 'data', 'top_class_dict.json')
BSD35K_FILTER_CSV  = os.path.join(PROJECT_ROOT, 'outputs', 'confidence_filter_v4', 'predictions', 'BSD35k-CS_filter_predictions_v4.csv')
BSD35K_AUDIO_DIR   = os.path.join(PROJECT_ROOT, 'data', 'features', 'BSD35k_clap_audio_embeddings')
BSD35K_TEXT_DIR    = os.path.join(PROJECT_ROOT, 'data', 'features', 'BSD35k-CS_clap_text_embeddings')
EXPERIMENTS_DIR    = os.path.join(PROJECT_ROOT, 'experiments')
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)

for p, name in [
    (PROCESSED_CSV,     'processed_dataset.csv'),
    (CLASS_DICT_JSON,   'class_dict.json'),
    (TOP_DICT_JSON,     'top_class_dict.json'),
    (BSD35K_FILTER_CSV, 'BSD35k-CS v4 filter predictions'),
    (BSD35K_AUDIO_DIR,  'BSD35k audio embeddings folder'),
    (BSD35K_TEXT_DIR,   'BSD35k-CS text embeddings folder'),
]:
    exists = '✓' if os.path.exists(p) else '✗ MISSING'
    print(f'  [{exists}] {name}')

with open(CLASS_DICT_JSON)  as f: class_dict     = json.load(f)
with open(TOP_DICT_JSON)    as f: top_class_dict = json.load(f)

# top-class → int index mapping (for loss)
class_to_top_idx = {}  # class_idx(int) → top_class_idx(int)
for cname, cidx in class_dict.items():
    top = cname.split('-')[0]
    class_to_top_idx[cidx] = top_class_dict[top]

# tensor for fast lookup
N = len(class_dict)
class_to_top_tensor = torch.zeros(N, dtype=torch.long)
for cidx, tidx in class_to_top_idx.items():
    class_to_top_tensor[cidx] = tidx
class_to_top_tensor = class_to_top_tensor.to(device)

print(f'\nClasses: {N} (2nd), {len(top_class_dict)} (top)')

In [ ]:
# ============================================================
# BSD35k-CS 필터링된 데이터셋 빌드
# ============================================================

filter_df = pd.read_csv(BSD35K_FILTER_CSV)
print(f'BSD35k-CS 전체: {len(filter_df)}개')

# v4 filter 적용
filtered = filter_df[filter_df['v4_filter_score'] >= BSD35K_FILTER_THRESHOLD].copy()
print(f'v4_filter_score >= {BSD35K_FILTER_THRESHOLD}: {len(filtered)}개 ({100*len(filtered)/len(filter_df):.1f}%)')

# class 매핑 (BSD10k class_dict 기준)
records = []
skipped_class  = 0
skipped_audio  = 0
skipped_text   = 0

for _, row in filtered.iterrows():
    cname = str(row['class'])

    # class_dict에 없는 클래스 (other, top-level 등) 제외
    if cname not in class_dict:
        skipped_class += 1
        continue

    sid = str(int(row['sound_id']))
    audio_path = os.path.join(BSD35K_AUDIO_DIR, f'{sid}.npy')
    text_path  = os.path.join(BSD35K_TEXT_DIR,  f'{sid}.npy')

    if not os.path.exists(audio_path):
        skipped_audio += 1
        continue
    if not os.path.exists(text_path):
        skipped_text += 1
        continue

    top = cname.split('-')[0]
    records.append({
        'index':            sid,
        'audio_emb_filepath': audio_path,
        'text_emb_filepath':  text_path,
        'top_class':          top,
        'top_class_idx':      top_class_dict[top],
        'class':              cname,
        'class_idx':          class_dict[cname],
    })

bsd35k_df = pd.DataFrame(records)
print(f'\n추가 가능 샘플: {len(bsd35k_df)}개')
print(f'  제외 (unknown class): {skipped_class}')
print(f'  제외 (audio missing): {skipped_audio}')
print(f'  제외 (text missing):  {skipped_text}')
print(f'\nBSD35k-CS 클래스 분포:')
print(bsd35k_df['class'].value_counts().sort_index())

In [ ]:
# ============================================================
# 계층적 손실함수 정의
# ============================================================

class HierarchicalLoss(nn.Module):
    """
    L_total = L_CE + lambda_top * L_Top + lambda_contr * L_Contr

    L_Top:   top-level misclassification penalty
    L_Contr: supervised contrastive loss (same top-class = positive pairs)
    """

    def __init__(self, class_to_top_tensor, lambda_top=0.3, lambda_contr=0.1, temperature=0.07):
        super().__init__()
        self.register_buffer('c2t', class_to_top_tensor)
        self.lambda_top   = lambda_top
        self.lambda_contr = lambda_contr
        self.temperature  = temperature
        self.ce = nn.CrossEntropyLoss(label_smoothing=0.01)

    def forward(self, logits, labels, z=None):
        # ── 1. Standard CE ──
        l_ce = self.ce(logits, labels)

        # ── 2. L_Top: top-class penalty ──
        preds     = torch.argmax(logits, dim=1)
        pred_top  = self.c2t[preds]
        true_top  = self.c2t[labels]
        l_top = (pred_top != true_top).float().mean()

        # ── 3. L_Contr: supervised contrastive (on latent z) ──
        l_contr = torch.tensor(0.0, device=logits.device)
        if z is not None and self.lambda_contr > 0:
            z_norm = F.normalize(z, dim=1)                      # (B, D)
            sim    = z_norm @ z_norm.T / self.temperature        # (B, B)

            top_labels = self.c2t[labels]                        # (B,)
            pos_mask = (top_labels.unsqueeze(0) == top_labels.unsqueeze(1))  # (B, B)
            pos_mask.fill_diagonal_(False)

            # mask out self from denominator
            eye = torch.eye(sim.size(0), dtype=torch.bool, device=sim.device)
            sim_exp = torch.exp(sim - sim.max(dim=1, keepdim=True).values)
            denom   = sim_exp.masked_fill(eye, 0).sum(dim=1, keepdim=True) + 1e-8

            log_prob = sim - torch.log(denom + 1e-8)

            n_pos = pos_mask.sum(dim=1).float()
            has_pos = n_pos > 0
            if has_pos.any():
                l_contr = -(log_prob[has_pos] * pos_mask[has_pos].float()).sum(dim=1)
                l_contr = (l_contr / n_pos[has_pos]).mean()

        total = l_ce + self.lambda_top * l_top + self.lambda_contr * l_contr
        return total, {'ce': l_ce.item(), 'top': l_top.item(), 'contr': l_contr.item() if isinstance(l_contr, torch.Tensor) else l_contr}

print('HierarchicalLoss 정의 완료')

In [ ]:
# ============================================================
# 학습 함수
# ============================================================

def make_serializable(obj, decimals=6):
    if isinstance(obj, torch.Tensor):
        return make_serializable(obj.detach().cpu().numpy(), decimals)
    elif isinstance(obj, np.ndarray):
        return round(float(obj), decimals) if obj.ndim == 0 else [make_serializable(x, decimals) for x in obj]
    elif isinstance(obj, float):
        return round(obj, decimals)
    elif isinstance(obj, int):
        return obj
    elif isinstance(obj, dict):
        return {k: make_serializable(v, decimals) for k, v in obj.items()}
    elif hasattr(obj, '__iter__') and not isinstance(obj, (str, bytes)):
        return [make_serializable(x, decimals) for x in obj]
    return obj


def train_one_fold(model, train_loader, val_loader, device,
                   criterion, num_epochs, lr, output_dir,
                   scheduler_type='step', patience=5, early_stopping_factor=3):
    os.makedirs(output_dir, exist_ok=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    if scheduler_type == 'step':
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    elif scheduler_type == 'plateau':
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=patience, verbose=False)
    else:
        scheduler = None

    best_acc  = 0.0
    no_improve = 0
    history    = defaultdict(list)

    for epoch in range(num_epochs):
        model.train()
        loss_sum, n = 0.0, 0
        loss_parts  = defaultdict(float)

        for data in train_loader:
            labels    = data['class_idx'].to(device)
            audio_emb = data.get('audio_embedding')
            text_emb  = data.get('text_embedding')
            if audio_emb is not None: audio_emb = audio_emb.to(device)
            if text_emb  is not None: text_emb  = text_emb.to(device)

            optimizer.zero_grad()
            z, logits, _ = model(audio_emb, text_emb)

            loss, parts = criterion(logits, labels, z)
            loss.backward()
            optimizer.step()

            bs       = labels.size(0)
            loss_sum += loss.item() * bs
            n        += bs
            for k, v in parts.items():
                loss_parts[k] += v * bs

        history['train_loss'].append(loss_sum / n)
        for k in loss_parts:
            history[f'train_{k}'].append(loss_parts[k] / n)

        # validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for data in val_loader:
                labels    = data['class_idx'].to(device)
                audio_emb = data.get('audio_embedding')
                text_emb  = data.get('text_embedding')
                if audio_emb is not None: audio_emb = audio_emb.to(device)
                if text_emb  is not None: text_emb  = text_emb.to(device)
                _, logits, _ = model(audio_emb, text_emb)
                correct += (torch.argmax(logits, 1) == labels).sum().item()
                total   += labels.size(0)

        val_acc = 100 * correct / total
        history['val_accuracy'].append(val_acc)

        ce   = loss_parts.get('ce',    0) / n
        top  = loss_parts.get('top',   0) / n
        con  = loss_parts.get('contr', 0) / n
        print(f'  Ep[{epoch+1:3d}] loss={loss_sum/n:.4f}  ce={ce:.4f} top={top:.4f} con={con:.4f} | val={val_acc:.2f}%')

        if scheduler:
            if scheduler_type == 'plateau': scheduler.step(val_acc)
            else: scheduler.step()

        if val_acc > best_acc:
            best_acc   = val_acc
            no_improve = 0
            cfg = {
                'hidden_size':   model.hidden_size,
                'num_classes':   model.num_classes,
                'emb_size_audio': model.emb_size_audio,
                'emb_size_text':  model.emb_size_text,
                'dropout':        model.dropout,
                'use_batch_norm': True,
                'mode':           model.mode,
            }
            torch.save({'model_state': model.state_dict(), 'config': cfg},
                       os.path.join(output_dir, 'best_model.pth'))
            print(f'    ★ best={val_acc:.2f}%  saved')
        else:
            no_improve += 1
            if no_improve >= patience * early_stopping_factor:
                print('  Early stopping.')
                break

    with open(os.path.join(output_dir, 'history.json'), 'w') as f:
        json.dump(make_serializable(history), f, indent=2)
    return best_acc, history, model

print('학습 함수 정의 완료')

In [ ]:
# ============================================================
# 5-FOLD 학습
# ============================================================

seed   = set_seed(SEED)
full_df = pd.read_csv(PROCESSED_CSV)
labels  = full_df['class_idx'].tolist()
skf     = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=seed)
class_to_top = build_class_to_topclass_mapping(class_dict, top_class_dict)

criterion = HierarchicalLoss(
    class_to_top_tensor,
    lambda_top=LAMBDA_TOP,
    lambda_contr=LAMBDA_CONTR,
    temperature=TEMPERATURE
).to(device)

fold_results = []

for fold, (trainval_idx, test_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f'\n{"+"*70}')
    print(f'  FOLD {fold} / {K_FOLDS-1}')
    print(f'{"+"*70}')

    # ── train/val split ──
    tv_labels = [labels[i] for i in trainval_idx]
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_rel, val_rel = next(sss.split(np.zeros(len(tv_labels)), tv_labels))
    train_idx = [trainval_idx[i] for i in train_rel]
    val_idx   = [trainval_idx[i] for i in val_rel]

    bsd10k_train = full_df.iloc[train_idx].reset_index(drop=True)
    val_df       = full_df.iloc[val_idx].reset_index(drop=True)
    test_df      = full_df.iloc[test_idx].reset_index(drop=True)

    print(f'BSD10k train: {len(bsd10k_train)}  val: {len(val_df)}  test: {len(test_df)}')
    print(f'BSD35k-CS 추가: {len(bsd35k_df)}')
    print(f'합계 train: {len(bsd10k_train) + len(bsd35k_df)}')

    # ── Dataset ──
    ds_10k   = HATRDataset(bsd10k_train, aug=True,  mask_pct=0.7)
    ds_35k   = HATRDataset(bsd35k_df,   aug=True,  mask_pct=0.7)   # BSD35k에도 augmentation
    ds_val   = HATRDataset(val_df,       aug=False)
    ds_test  = HATRDataset(test_df,      aug=False)

    train_combined = ConcatDataset([ds_10k, ds_35k])

    train_loader = DataLoader(train_combined, batch_size=BATCH_SIZE, shuffle=True,
                              drop_last=True,  num_workers=0, pin_memory=torch.cuda.is_available())
    val_loader   = DataLoader(ds_val,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=torch.cuda.is_available())
    test_loader  = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=torch.cuda.is_available())

    # ── 모델 ──
    emb_audio = 512 if MODE in ['audio', 'both'] else 0
    emb_text  = 512 if MODE in ['text',  'both'] else 0

    model = BaseClassifier(
        hidden_size=HIDDEN_SIZE,
        num_classes=len(class_dict),
        emb_size_audio=emb_audio,
        emb_size_text=emb_text,
        dropout=DROPOUT,
        use_batch_norm=True,
        mode=MODE
    ).to(device)

    # Xavier init
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)

    output_dir = os.path.join(PROJECT_ROOT, MODEL_OUTPUT_DIR, MODE, f'fold_{fold}')

    best_acc, history, trained = train_one_fold(
        model, train_loader, val_loader, device,
        criterion=criterion,
        num_epochs=NUM_EPOCHS,
        lr=LR,
        output_dir=output_dir,
        scheduler_type=SCHEDULER_TYPE,
        patience=PATIENCE,
        early_stopping_factor=EARLY_STOPPING_FACTOR
    )

    # ── 평가 ──
    model_path = os.path.join(output_dir, 'best_model.pth')
    metrics = evaluate_model(
        BaseClassifier, model_path, test_loader, device,
        class_to_top, output_dir=output_dir, fold_id=fold, class_dict=class_dict
    )

    fold_results.append({
        'fold':           fold,
        'train_size':     len(bsd10k_train) + len(bsd35k_df),
        'bsd10k_size':    len(bsd10k_train),
        'bsd35k_size':    len(bsd35k_df),
        'test_size':      len(test_df),
        'accuracy':       metrics['accuracy'],
        'top_accuracy':   metrics['top_accuracy'],
        'macro_accuracy': metrics['macro_accuracy'],
        'h_acc':          metrics['hierarchical_accuracy'],
        'h_f1':           metrics['hierarchical_f1'],
    })

    print(f'\n── Fold {fold} 결과 ──')
    print(f'  H-Acc:     {metrics["hierarchical_accuracy"]:.2f}%')
    print(f'  Top Acc:   {metrics["top_accuracy"]:.2f}%')
    print(f'  Macro 2nd: {metrics["macro_accuracy"]:.2f}%')
    print(f'  H-F1:      {metrics["hierarchical_f1"]:.2f}%')

print('\n모든 fold 완료!')

In [ ]:
# ============================================================
# 결과 집계 + baseline 비교
# ============================================================

BASELINE = {
    'accuracy':       79.63,
    'top_accuracy':   88.88,
    'macro_accuracy': 74.02,
    'h_acc':          79.45,
    'h_f1':           78.33,
}

keys = ['accuracy', 'top_accuracy', 'macro_accuracy', 'h_acc', 'h_f1']
avg  = {k: np.mean([r[k] for r in fold_results]) for k in keys}

print('\n' + '='*65)
print('EXP-003+005 결과 요약')
print('='*65)
labels_map = {
    'accuracy':       'Accuracy (micro)',
    'top_accuracy':   'Top Accuracy',
    'macro_accuracy': 'Macro 2nd',
    'h_acc':          'H-Acc ★ (랭킹 지표)',
    'h_f1':           'H-F1',
}
print(f"{'Metric':<24} {'Baseline':>10} {'EXP-003':>10} {'Diff':>8}")
print('-'*65)
for k in keys:
    d = avg[k] - BASELINE[k]
    arrow = '↑' if d > 0 else ('↓' if d < 0 else '=')
    print(f"{labels_map[k]:<24} {BASELINE[k]:>9.2f}% {avg[k]:>9.2f}% {arrow}{abs(d):>5.2f}%")
print('='*65)

print(f'\nFold별 H-Acc:')
for r in fold_results:
    print(f"  Fold {r['fold']} (train={r['train_size']}): {r['h_acc']:.2f}%")
print(f"  평균: {avg['h_acc']:.2f}%  vs  baseline: {BASELINE['h_acc']:.2f}%")

In [ ]:
# ============================================================
# 실험 결과 JSON 저장
# ============================================================

try:
    git_commit = subprocess.check_output(
        ['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_ROOT
    ).decode().strip()
except:
    git_commit = 'unknown'

vs_base = f"{avg['h_acc'] - BASELINE['h_acc']:+.2f}%"

result_json = {
    'exp_id':      EXP_ID,
    'branch':      'exp/bsd35k-hier-loss',
    'description': EXP_DESC,
    'config': {
        'seed':                   SEED,
        'hidden_size':            HIDDEN_SIZE,
        'mode':                   MODE,
        'epochs':                 NUM_EPOCHS,
        'batch_size':             BATCH_SIZE,
        'lr':                     LR,
        'dropout':                DROPOUT,
        'bsd35k_filter_threshold': BSD35K_FILTER_THRESHOLD,
        'bsd35k_samples_added':   len(bsd35k_df),
        'lambda_top':             LAMBDA_TOP,
        'lambda_contr':           LAMBDA_CONTR,
        'temperature':            TEMPERATURE,
        'loss':                   'CE + L_Top + L_Contr',
    },
    'results': {
        'fold_avg': {
            'macro_acc_2nd':    round(avg['macro_accuracy'], 4),
            'macro_acc_top':    round(avg['top_accuracy'],   4),
            'hierarchical_acc': round(avg['h_acc'],         4),
            'hierarchical_f1':  round(avg['h_f1'],          4),
        },
        'fold_details': [
            {
                'fold':         r['fold'],
                'train_size':   r['train_size'],
                'bsd35k_added': r['bsd35k_size'],
                'test_size':    r['test_size'],
                'accuracy':     round(r['accuracy'],       4),
                'top_accuracy': round(r['top_accuracy'],   4),
                'macro_acc':    round(r['macro_accuracy'], 4),
                'h_acc':        round(r['h_acc'],          4),
                'h_f1':         round(r['h_f1'],           4),
            }
            for r in fold_results
        ]
    },
    'vs_baseline_h_acc': vs_base,
    'baseline_h_acc':    BASELINE['h_acc'],
    'timestamp':         datetime.now().strftime('%Y-%m-%d %H:%M'),
    'git_commit':        git_commit,
}

out_path = os.path.join(EXPERIMENTS_DIR, f'{EXP_ID}_bsd35k_hier.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(result_json, f, indent=2, ensure_ascii=False)

print(f'저장: {out_path}')
print(f'\n최종 H-Acc: {avg["h_acc"]:.2f}%  (baseline: {BASELINE["h_acc"]:.2f}%  →  {vs_base})')